In [0]:
# To Know the Scope Name

display(dbutils.secrets.listScopes())

In [0]:
# to know Secret keys

dbutils.secrets.list("azure-kv-scope")

In [0]:
storageaccountname = "adlsdatainfra"
containername = "raw"
clientid = dbutils.secrets.get(scope = "azure-kv-scope",key = "GenZ-Vault-AppID")
clientsecret = dbutils.secrets.get(scope = "azure-kv-scope",key = "GenZ-Vault-AppPassword")
tenantID = dbutils.secrets.get(scope = "azure-kv-scope",key = "GenZ-Vault-TenantID")

spark.conf.set("fs.azure.account.auth.type." + storageaccountname + ".dfs.core.windows.net", "OAuth")

spark.conf.set("fs.azure.account.oauth.provider.type." + storageaccountname + ".dfs.core.windows.net",
                "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")

spark.conf.set("fs.azure.account.oauth2.client.id." + storageaccountname + ".dfs.core.windows.net", clientid)
spark.conf.set("fs.azure.account.oauth2.client.secret." + storageaccountname + ".dfs.core.windows.net", clientsecret)
spark.conf.set("fs.azure.account.oauth2.client.endpoint." + storageaccountname + ".dfs.core.windows.net", 
               "https://login.microsoftonline.com/" + tenantID + "/oauth2/v2.0/token")


In [0]:
dbutils.fs.ls("abfss://raw@adlsdatainfra.dfs.core.windows.net/")

In [0]:
from pyspark.sql.functions import current_timestamp, col

df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("abfss://raw@adlsdatainfra.dfs.core.windows.net/AirBnb.csv")
    .select("*", "_metadata")
)

df = (
    df.withColumn("source_file", col("_metadata.file_path"))
      .withColumn("ingestion_timestamp", current_timestamp())
      .drop("_metadata")
)

display(df)

In [0]:
df.printSchema()

In [0]:
df.count()

In [0]:
df.show(5)

In [0]:
from pyspark.sql.functions import *

null_df = df.select([
    count(
        when(
            col(column).isNull() | (trim(col(column).cast("string")) == ""),
            column
        )
    ).alias(column)
    for column in df.columns
])

display(null_df)

### CATALOG

In [0]:
%sql
CREATE CATALOG airbnb_analytics;

USE CATALOG airbnb_analytics;

CREATE SCHEMA bronze;
CREATE SCHEMA silver;
CREATE SCHEMA gold;

In [0]:
# Write Data to Bronze Delta Table

df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("airbnb_analytics.bronze.airbnb_listings")

In [0]:
%sql
SELECT COUNT(*)
FROM airbnb_analytics.bronze.airbnb_listings;

In [0]:
%sql
SELECT *
FROM airbnb_analytics.bronze.airbnb_listings
LIMIT 10;

### Metadata Check

In [0]:
%sql
SELECT source_file,
       ingestion_timestamp
FROM airbnb_analytics.bronze.airbnb_listings
LIMIT 10;